In [3]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target)

with mlflow.start_run():
    model = RandomForestClassifier(n_estimators=50)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", 50)
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(model, "model")

    print("Accuracy:", acc)

/home/sigma_867/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/01 20:39:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/01 20:39:51 INFO mlflow.store.db.utils: Updating database tables
2026/04/01 20:39:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 20:39:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/sigma_867/.local/lib/python3.14/site-packages/langchain_core/_api

Accuracy: 0.9210526315789473


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ---------------------------
# STEP 1: DEFINE TEAM DATA
# ---------------------------

teams = {
    "Senegal": {"tier": 3, "elo": 1750, "goals": 1.8, "conceded": 0.9},
    "Morocco": {"tier": 3, "elo": 1800, "goals": 1.6, "conceded": 0.8},
    "Ivory Coast": {"tier": 3, "elo": 1720, "goals": 1.5, "conceded": 1.0},
    
    "Egypt": {"tier": 2, "elo": 1680, "goals": 1.4, "conceded": 0.9},
    "Algeria": {"tier": 2, "elo": 1700, "goals": 1.6, "conceded": 1.1},
    "DRC": {"tier": 2, "elo": 1600, "goals": 1.2, "conceded": 1.2},
    "Cape Verde": {"tier": 2, "elo": 1580, "goals": 1.3, "conceded": 1.1},
    
    "Ghana": {"tier": 1, "elo": 1550, "goals": 1.3, "conceded": 1.3},
    "RSA": {"tier": 1, "elo": 1500, "goals": 1.1, "conceded": 1.2},
    "Tunisia": {"tier": 1, "elo": 1620, "goals": 1.2, "conceded": 1.0}
}

# ---------------------------
# STEP 2: GENERATE MATCH DATA
# ---------------------------

def generate_matches(teams):
    data = []
    team_names = list(teams.keys())
    
    for i in range(len(team_names)):
        for j in range(i+1, len(team_names)):
            t1 = teams[team_names[i]]
            t2 = teams[team_names[j]]
            
            features = {
                "tier_diff": t1["tier"] - t2["tier"],
                "elo_diff": t1["elo"] - t2["elo"],
                "goal_diff": t1["goals"] - t2["goals"],
                "defense_diff": t2["conceded"] - t1["conceded"]
            }
            
            # Simulated outcome (based on ELO)
            if t1["elo"] > t2["elo"] + 50:
                result = 2  # win
            elif t1["elo"] < t2["elo"] - 50:
                result = 0  # loss
            else:
                result = 1  # draw
            
            features["result"] = result
            data.append(features)
    
    return pd.DataFrame(data)

df = generate_matches(teams)

# ---------------------------
# STEP 3: TRAIN MODEL
# ---------------------------

X = df.drop("result", axis=1)
y = df["result"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# ---------------------------
# STEP 4: EVALUATE
# ---------------------------

preds = model.predict(X_test)
print(classification_report(y_test, preds))

# ---------------------------
# STEP 5: PREDICT NEW MATCH
# ---------------------------

def predict_match(team1, team2):
    t1 = teams[team1]
    t2 = teams[team2]
    
    sample = pd.DataFrame([{
        "tier_diff": t1["tier"] - t2["tier"],
        "elo_diff": t1["elo"] - t2["elo"],
        "goal_diff": t1["goals"] - t2["goals"],
        "defense_diff": t2["conceded"] - t1["conceded"]
    }])
    
    prediction = model.predict(sample)[0]
    
    outcomes = ["Loss", "Draw", "Win"]
    return outcomes[prediction]

# Example
print("Senegal vs Ghana:", predict_match("Senegal", "Ghana"))

              precision    recall  f1-score   support

           1       0.67      1.00      0.80         2
           2       1.00      0.86      0.92         7

    accuracy                           0.89         9
   macro avg       0.83      0.93      0.86         9
weighted avg       0.93      0.89      0.90         9

Senegal vs Ghana: Win
